In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import URL
import os
from dotenv import load_dotenv

### 为什么这样创建数据库引擎？

直接把连接地址、用户名和密码写成一行，也可以正常连接数据库。这里使用 `.env` 和 `URL.create()`，是为了方便项目公开、配置维护和正确处理连接信息，不是为了提高查询速度。

1. **让密码与公开代码分开。** 本项目计划用于 GitHub 作品集，不能把数据库密码直接写进 notebook。`.env` 保存本机连接配置，代码只读取配置。需要确保 `.env` 没有被 Git 跟踪，并在 `.gitignore` 中排除它。`.env` 本身不加密，也不会自动清除以前提交过的密码。
2. **多处连接共用一份配置。** 项目中的连接检查脚本、数据导入脚本和 notebook 都可以读取同一份 `.env`。修改数据库地址、用户名或密码时，只需更新配置并重新加载、创建引擎，不必逐个修改分析文件中的连接字符串。其他人复现时也可以填写自己的数据库配置。
3. **正确处理密码中的特殊字符。** 手工拼接连接字符串时，密码里的 `@`、`/` 等字符可能被当成地址结构的一部分，需要额外编码。`URL.create()` 接收独立的用户名、密码、主机等参数，避免手工处理这些转义问题。

下面几部分各自负责什么：

- `load_dotenv(...)`：把指定 `.env` 文件中的配置加载到当前 Python 进程的环境变量中；默认不会覆盖已经存在的同名环境变量。
- `os.getenv(...)`：读取对应配置值，本身不负责打开 `.env` 文件。
- `URL.create(...)`：把这些配置组合成 SQLAlchemy 使用的连接信息，`utf8mb4` 与当前项目的字符集设置保持一致。
- `create_engine(...)`：创建数据库连接引擎，供后面的 `pd.read_sql(...)` 使用。创建引擎通常还没有实际连接数据库；后续读取数据成功，才能说明连接和查询都已成功。

**当前路径说明：** 示例里的 `D:\my_project\ecommerce-business-analytics\.env` 是这台电脑的绝对路径。换电脑或移动项目时，需要调整这个路径；读取 `.env` 并不意味着代码已经可以在任何目录直接复现。


In [2]:
# 创建引擎对象
load_dotenv(r"D:\my_project\ecommerce-business-analytics\.env")
engine = create_engine(URL.create("mysql+pymysql",
                                  username=os.getenv("DB_USER"),
                                  password=os.getenv("DB_PASSWORD"),
                                  host=os.getenv("DB_HOST"),
                                  port=int(os.getenv("DB_PORT", "3306")),
                                  database=os.getenv("DB_NAME"),
                                  query={"charset": "utf8mb4"}
                                  ))

In [3]:
data = pd.read_sql('v_sales_transactions', engine)
data

,raw_id,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet,source_row
0,1,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,2
1,2,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,3
2,3,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,4
3,4,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,5
4,5,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,6
...,...,...,...,...,...,...,...,...,...,...,...
1015083,1067366,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,Year 2010-2011,541906
1015084,1067367,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,Year 2010-2011,541907
1015085,1067368,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011,541908
1015086,1067369,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,Year 2010-2011,541909


In [4]:
clean = data.query('invoice_date >= "2011-11-01" & invoice_date < "2011-12-01" & customer_id.notna()')
clean

,raw_id,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,source_sheet,source_row
906965,957135,573744,21314,SMALL GLASS HEART TRINKET POT,8,2011-11-01 08:16:00,2.10,17733.0,United Kingdom,Year 2010-2011,431675
906966,957136,573744,21704,BAG 250g SWIRLY MARBLES,12,2011-11-01 08:16:00,0.85,17733.0,United Kingdom,Year 2010-2011,431676
906967,957137,573744,21791,VINTAGE HEADS AND TAILS CARD GAME,12,2011-11-01 08:16:00,1.25,17733.0,United Kingdom,Year 2010-2011,431677
906968,957138,573744,21892,TRADITIONAL WOODEN CATCH CUP GAME,12,2011-11-01 08:16:00,1.25,17733.0,United Kingdom,Year 2010-2011,431678
906969,957139,573744,21915,RED HARMONICA IN BOX,12,2011-11-01 08:16:00,1.25,17733.0,United Kingdom,Year 2010-2011,431679
...,...,...,...,...,...,...,...,...,...,...,...
990052,1041826,579885,85034C,3 ROSE MORRIS BOXED CANDLES,4,2011-11-30 17:37:00,1.25,15444.0,United Kingdom,Year 2010-2011,516366
990053,1041827,579885,21742,LARGE ROUND WICKER PLATTER,2,2011-11-30 17:37:00,5.95,15444.0,United Kingdom,Year 2010-2011,516367
990054,1041828,579885,23084,RABBIT NIGHT LIGHT,6,2011-11-30 17:37:00,2.08,15444.0,United Kingdom,Year 2010-2011,516368
990055,1041829,579885,21257,VICTORIAN SEWING BOX MEDIUM,1,2011-11-30 17:37:00,7.95,15444.0,United Kingdom,Year 2010-2011,516369


In [5]:
purchase = clean.groupby(['customer_id','stock_code'])['invoice'].nunique().reset_index(name= 'purchase_count')
# 方法1： filter函数过滤
new = purchase.filter(lambda x: x['purchase_count'] >= 2)
# 方法2： query函数筛选
new1 = purchase.query('purchase_count >= 2')
new1
len(new)
len(new1)

4629

In [6]:
new['customer_id'].nunique()

412